# Zoom動画 自動編集（Google Colab版）

Zoomの録画を、**OP/EDカット → 登壇者の開始点で開始 → 無音カット → 登壇者以外の発話カット →（任意）AIで「もたつき・トラブル」区間を検出してカット → OP/ED素材をディゾルブで接続** まで自動で編集し、mp4を書き出すノートブックです。
書き出しには Colab の **無料GPU** が使えます（使える場合は NVENC で高速に処理します）。

コード本体は GitHub リポジトリ（KamikiKei/zoom-video-auto-editor）から毎回取得するので、**このノートブック1枚を配れば、コードは常に最新**になります。

## 準備するもの（Google Drive に置いてください）

| もの | 備考 |
|---|---|
| 録画ファイル（.mp4） | Zoomの録画本体 |
| Zoom文字起こし（.vtt） | Zoomのクラウド録画の「音声文字起こし」。**話者カットとAI検出に必要**です。無い場合は開始点の自動推定・話者カット・AI検出が働きません |
| OP/ED素材 | Canva等で作った素材。画像（png/jpg）または動画（mp4等）。頭とお尻に同じ素材を使います |

おすすめの置き方： マイドライブに `video-auto-editor` フォルダを作り、その中に `input`（材料）と `output`（成果物）のフォルダを作る。

## 実行順序

上のセルから順番に実行してください（Shift+Enter で次のセルへ）。

1. 環境確認（GPU / ffmpeg）
2. コード取得（GitHubから clone / 更新）
3. Google Drive マウント
4. 設定（ファイルの場所などを指定）＋ Gemini APIキーの読み込み（任意）
5. plan（カット案づくり）
6. render（書き出し）
7. 結果確認

## 注意

- **GPUランタイム**にしてください： メニュー「ランタイム」→「ランタイムのタイプを変更」→ ハードウェアアクセラレータで **GPU（T4など）** を選択。無料枠のGPUは割り当て状況次第で、使えない時間帯もあります（その場合はCPU書き出しになりますが処理はできます。少し待ちます）。
- **セッションが切れても、出力ファイルは Drive に残ります**（出力先を Drive にしているため）。途中で切れた場合は、もう一度上から実行し直してください。plan（カット案）も Drive の出力フォルダに保存されるので、render だけ再実行することもできます。
- Gemini APIキー（AI検出用）は、ノートブックに直接書かず **Colab の Secrets** に登録してください（手順は設定セルのコメント参照）。キーが無い場合は AI検出なしでそのまま進みます。

## 1. 環境確認

GPUの有無と、ffmpeg で `h264_nvenc`（GPUエンコード）が使えるかを確認します。

In [ ]:
# GPUの確認（GPUランタイムでない場合は何も表示されません）
!nvidia-smi

# ffmpeg の確認
!ffmpeg -version | head -2

# NVENC が実際に使えるか、小さい映像で試しエンコードして確認
import subprocess
r = subprocess.run(
    ["ffmpeg", "-y", "-f", "lavfi", "-i", "testsrc=size=256x256:rate=30:duration=0.5",
     "-c:v", "h264_nvenc", "-f", "null", "-"],
    capture_output=True, text=True)
print("h264_nvenc:", "使えます（GPUで高速に書き出します）" if r.returncode == 0
      else "使えません（CPUの libx264 で書き出します）")

## 2. コード取得

GitHub リポジトリを取得します（2回目以降は最新に更新します）。
このツール（plan / render）は Python 標準ライブラリだけで動くので、追加のインストールは不要です。

In [ ]:
%cd /content
!if [ -d zoom-video-auto-editor ]; then cd zoom-video-auto-editor && git pull; else git clone https://github.com/KamikiKei/zoom-video-auto-editor.git; fi
%cd /content/zoom-video-auto-editor

## 3. Google Drive マウント

許可ダイアログが出たら、Drive へのアクセスを許可してください。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. 設定

材料ファイルの場所と、出力先を指定します。**自分の Drive の構成に合わせて書き換えてください。**
ついでに、AI検出用の Gemini APIキーを Colab の Secrets から読み込みます。

### Gemini APIキーの登録方法（AI検出を使いたい人だけ）

1. 左端の鍵アイコン（Secrets）を開く
2. 「新しいシークレットを追加」で、名前を `GEMINI_API_KEY` にして、値にAPIキーを貼る
3. 「ノートブックからのアクセス」を ON にする

キーが登録されていない場合は、AI検出なしでそのまま進みます（話者カットや無音カットは動きます）。

In [ ]:
import os

# ===== ここを自分の環境に合わせて書き換える =====
BASE = "/content/drive/MyDrive/video-auto-editor"
INPUT_VIDEO   = f"{BASE}/input/recording.mp4"   # 録画 mp4
TRANSCRIPT_VTT = f"{BASE}/input/recording.vtt"  # Zoom文字起こし VTT（無い場合は "" にする）
ASSET_FILE    = f"{BASE}/input/op_ed.mp4"       # OP/ED素材（動画 or 画像 png/jpg）
OUTPUT_DIR    = f"{BASE}/output"                # 出力先フォルダ（Drive上）
OUTPUT_NAME   = "edited.mp4"                    # 出力ファイル名

SILENCE_CUT   = True   # 無音カットをするか
SILENCE_SEC   = 3.0    # 何秒以上の無音をカットするか
PRESENTERS    = ""     # 登壇者の話者番号（例 "0" や "0,1"）。空欄なら自動推定
ENCODER       = "auto" # auto / x264 / nvenc（autoならNVENCが使える環境ではNVENC）
# ===============================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
PLAN_PATH = f"{OUTPUT_DIR}/plan.json"
OUTPUT_PATH = f"{OUTPUT_DIR}/{OUTPUT_NAME}"

for label, p in [("動画", INPUT_VIDEO), ("素材", ASSET_FILE)]:
    print(("OK  " if os.path.exists(p) else "無い ") + f"{label}: {p}")
if TRANSCRIPT_VTT:
    print(("OK  " if os.path.exists(TRANSCRIPT_VTT) else "無い ")
          + f"文字起こし: {TRANSCRIPT_VTT}（無いと話者カット・AI検出が働きません）")

# Gemini APIキー（Colab Secrets から。キーの値をこのノートブックに書かないこと）
try:
    from google.colab import userdata
    _key = userdata.get("GEMINI_API_KEY")
    if _key:
        os.environ["GEMINI_API_KEY"] = _key
        print("Gemini APIキー: 読み込みました（AI検出を使います）")
    else:
        print("Gemini APIキー: 未設定のため、AI検出なしで進みます")
except Exception:
    print("Gemini APIキー: 未設定のため、AI検出なしで進みます")

## 5. plan（カット案づくり）

カット案を作ります（まだ書き出しはしません）。
開始点・カット候補（無音／運営など登壇者以外の発話／AI検出）・想定の出力尺が表示されます。
おかしいところがあれば、設定セルに戻って直してから実行し直してください。

In [ ]:
import json, subprocess, sys

cmd = [sys.executable, "edit.py", "plan",
       "--video", INPUT_VIDEO,
       "--thumb", ASSET_FILE,
       "--out", PLAN_PATH]
if TRANSCRIPT_VTT:
    cmd += ["--transcript", TRANSCRIPT_VTT]
    cmd += ["--llm-detect"]  # キーが無ければ自動でスキップされます
if SILENCE_CUT:
    cmd += ["--silence-cut", "--silence-sec", str(SILENCE_SEC)]
if PRESENTERS:
    cmd += ["--presenters", PRESENTERS]

r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError("plan の作成に失敗しました")

plan = json.load(open(PLAN_PATH, encoding="utf-8"))

def mmss(sec):
    sec = int(round(sec)); return f"{sec // 60}:{sec % 60:02d}"

cands = plan["silence"]["candidates"]
print("=" * 50)
print(f"元の尺　　　: {mmss(plan['video']['duration'])}")
print(f"開始点　　　: {mmss(plan['trim_start'])}（{plan['trim_start_source']}）")
print(f"終了点　　　: {mmss(plan['end'])}")
print(f"想定の出力尺: {mmss(plan['expected_duration'])}")
print(f"登壇者　　　: {plan['presenters']}（保持ポリシー適用: {plan['keep_policy']['applied']}）")
from collections import Counter
by_src = Counter(c["source"] for c in cands if c["enabled"])
label = {"silence": "無音", "ops": "運営の発話", "other": "登壇者以外の発話",
         "llm": "AI検出", "user": "手動"}
print("カット候補（有効なもの）:")
for src, n in by_src.items():
    print(f"  {label.get(src, src)}: {n}件")
llm = plan.get("llm_detection") or {}
print(f"AI検出　　　: {llm.get('status')} {llm.get('detail', '')}")
print(f"plan保存先　: {PLAN_PATH}")

## 6. render（書き出し）

plan をもとに書き出します。エンコーダは `auto` の場合、NVENC が使える環境では NVENC が選ばれます。
実際に使われたエンコーダと所要時間を表示します。出力は Drive の出力フォルダに保存されます。

In [ ]:
import re, subprocess, sys, time

cmd = [sys.executable, "edit.py", "render",
       "--plan", PLAN_PATH,
       "--thumb", ASSET_FILE,
       "-o", OUTPUT_PATH,
       "--encoder", ENCODER]

t0 = time.time()
r = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.time() - t0
print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError("書き出しに失敗しました")

m = re.search(r"エンコーダ: (\S+)", r.stdout)
print("=" * 50)
print(f"使われたエンコーダ: {m.group(1) if m else '不明（上のログを確認）'}")
print(f"所要時間　　　　　: {elapsed / 60:.1f} 分")
print(f"出力　　　　　　　: {OUTPUT_PATH}")

## 7. 結果確認

出力ファイルの尺とサイズを表示します。ファイルは Drive の出力フォルダにあります。

In [ ]:
import os, sys
sys.path.insert(0, "/content/zoom-video-auto-editor")
from edit import probe  # ffmpeg の情報から尺を読む（このツールの関数を流用）

info = probe(OUTPUT_PATH)
dur = float(info["format"]["duration"])
size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"出力ファイル: {OUTPUT_PATH}")
print(f"尺　　　　　: {int(dur // 60)}:{int(dur % 60):02d}")
print(f"サイズ　　　: {size_mb:.1f} MB")